# Sliding-window reader scalp-map tests — recovered ZuCo
**CPU → Run all**, same Zyion Drive account. Reads only saved per-reader sufficient statistics (roughly tens of MB), **not individual EEG archives**. No re-extraction, cleaning, training or GPU required. New output capped at 10 MB in a separate folder.

Fixed **80 ms endpoint span**, moved by **8 ms** across the full −600..1000ms epoch. Each window includes 11 samples at 125 Hz, so 224..304 ms is included exactly. Raw and cleaned signals are tested together:191 windows each,382 comparisons. All 4096 sign flips of the 12 reader contrasts are enumerated, shared across all windows and both representations.

Each reader contributes its mean paired positive−negative scalp map. Remove the spatial mean and compute the magnitude of the across-reader mean contrast map. A sign-invariant per-window scale standardizes the statistic for the across-window maximum. Marginal p-values match the existing per-window reader test; corrected p-values use the single largest statistic across the entire scan. This is not original TCT/TANOVA, does not locate a significant electrode cluster, and does not isolate sentiment from word-frequency/context confounds.

This is **exploratory on already-inspected data**. Overlapping windows are correlated. Do not interpret a significant window center as an exact neural onset. Do not choose another width/step after looking at results and call it the same corrected family.


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
RUN=THESIS/'Results/word_sentiment_eeg/zuco_paper_figures_recovered'
OUT=THESIS/'Results/word_sentiment_eeg/zuco_sliding_reader_maps'
REPO=Path('/content/word-sentiment-eeg-sliding-maps')
CODE_REVISION='0fa726d19d50acb6f8b1aae5f6a8029563701850'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# A mounted Drive filesystem disk_usage value can reflect the runtime volume;
# use the authenticated Drive quota instead.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();quota=response.json()['storageQuota']
print('Quota account:',response.json().get('user',{}).get('emailAddress','unavailable'))
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a finite storage quota; verify free space before starting.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
print(f'Account free space: {FREE_DRIVE_BYTES/1e9:.2f} GB. Workflow checks the additional space needed before source reads.')
print('Destination:',OUT)

used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES < max(0,10_000_000-used):
    raise RuntimeError('Need up to 10 MB free for all new follow-up outputs; existing results unchanged.')


## Run the fixed scan

In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/zuco_sliding_reader_maps.py'),'--thesis-root',str(THESIS)]
run_with_progress(command,Path('/content/sliding-reader-maps-status.json'),activity='Sliding reader-map scan: saved small statistics only')


## Scan plot and corrected significant windows

In [ ]:
display(Image(filename=str(OUT/'sliding_reader_maps.png')))
result=json.loads((OUT/'window_results.json').read_text())
windows=pd.DataFrame(result['rows'])
print('Windows per representation:',result['windows_per_representation'],'   exact sign assignments:',result['assignments'])
passed=windows[windows.significant]
if passed.empty:print('No window passed the correction across the complete scan.')
else:display(passed)
print('Smallest corrected p per representation:')
display(windows.groupby('representation')[['p_uncorrected','p_familywise']].min())
print('All windows remain in:',OUT/'window_results.csv')


## Published 224–304 ms window, evaluated inside the full scan

In [ ]:
display(windows[windows.start_ms.eq(224)&windows.end_ms.eq(304)])
print('These corrected p-values can be larger than the previous correction over four fixed tests, because this scan covers382 comparisons.')
print('Saved results:',OUT)
